# 17 - How much does the assumed burn error set the delivery error? (Kaggle)

Notebook 16 flew seeds 0-149 from the real Checkpoint (125 m) and found a median delivery error of 0.68 m, against 0.39 m from 45 m. The explanation is that the Matchpoint burn is realistically large (0.07 m/s) and its **assumed** execution errors (1% magnitude, 0.5 degree pointing, 0.2 mm/s fixed) act over the roughly 790 s arc. This notebook tests that: the same seeds 0-49, no detector, from 125 m, with the burn errors **at zero** (perfect burns) and **doubled**. Together with the nominal run in notebook 16 that gives three points. Seeds share everything random, so the comparison is paired (`scripts/compare_start_alt.py DIR_A DIR_B` works on any two folders).

Settings: **GPU T4 x2**, **Internet on**. No dataset needed. Save Version, Save & Run All (Commit). Two flight processes run side by side (50 landings each); about 3.5 hours.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
# the 195 MB global model, then the wide terrain (writes /tmp/data/nightingale_wide_5cm.npz, about 15 s)
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/fetch_data.py --global
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/build_wide_dtm.py --out-dir /tmp/data

In [ ]:
import subprocess
W = '/kaggle/working/burn_sens'
# no detector: --ckpt is omitted
run = 'python /tmp/touch_down/scripts/kaggle_run.py --max-hours 6 --wide --seeds 0 50'
procs = [subprocess.Popen(f"{run} --burn-error-scale {k} --out {W}/scale{k}", shell=True) for k in (0.0, 2.0)]
for p in procs:
    p.wait()
print('both flight processes finished', [p.returncode for p in procs])